# Define Persistence

Persistence means something that continues to exist or is saved even after the program stops or restarts.

# Layman Example (Real Life)

Imagine you have a notebook.

You write your homework in the notebook.
You close the notebook.

Next day, when you open it, everything is still there.

➡️ This is persistence.

# Now imagine writing on a whiteboard.

You write something.

Someone wipes it clean.

Everything is gone.

➡️ This is not persistence.

# Persistence means data is stored permanently so it doesn't disappear when the application shuts down.

# In LangGraph, persistence means saving the graph's state (memory) so the workflow can pause, resume, recover from failures, or continue later without losing progress.

# In persistence it not only store final vlaue but also store a intermediate value also in state

# Other Reasons for Storing Intermediate State
* Fault tolerance: Recover after crashes.

* Human-in-the-loop: Pause the workflow, let a human review or approve, then resume.

* Long-running tasks: Some workflows take minutes, hours, or even days.

* Debugging: You can inspect what each node produced if something goes wrong.

* Avoid repeating expensive work: Don't rerun database queries or LLM calls unnecessarily.

In [ ]:
# Visual Representation
Node 1
   │
   ▼
State 1 (saved)
   │
   ▼
Node 2
   │
   ▼
State 2 (saved)
   │
   ▼
Node 3
   │
   ▼
State 3 (saved)
   │
   ▼
Node 4
   │
   ▼
Final Output

If the system crashes after State 3, it resumes from Node 4, not Node 1.

# Checkpointer in Persistence

checkpointer is the component that saves and restores the state of your LangGraph workflow.

# Real-Life Example (Video Game 🎮)

Imagine you're playing GTA.

Without saving:

Level 1
   ↓
Level 2
   ↓
Level 3
   ↓
💥 Game crashes

You start again from Level 1.

With a save point (checkpoint):

Level 1
   ↓
💾 Checkpoint
   ↓
Level 2
   ↓
💾 Checkpoint
   ↓
Level 3
   ↓
💥 Game crashes

When you restart:

Resume from Level 3 ✅

The checkpoint saved your progress.

# In LangGraph

Suppose your graph looks like this:

User Input
     │
     ▼
Understand Query
     │
     ▼
Search Database
     │
     ▼
LLM Generates Answer
     │
     ▼
Return Response

A checkpointer saves the state after important nodes.

In [ ]:
# What Does a Checkpointer Do?

It has two main jobs:

# Save state

After a node finishes, save the current state.

# Load state

If the workflow restarts, load the last saved state and continue.

# Checkpointer ek "save button" ki tarah hota hai jo workflow ki progress ko baar-baar save karta rehta hai, taaki agar system crash ho jaye ya workflow pause ho jaye, to wahi se dobara continue kiya ja sake.

# Thread in Persistence

Layman Hinglish Explanation

Imagine WhatsApp. 📱

Tumhare alag-alag logon ke saath alag chats hoti hain.

Rahul Chat
------------
Hi
How are you?

Aman Chat
------------
Hello

Mom Chat
------------
Did you eat?

Har chat alag hai.

Rahul wali chat mein jo messages hain, woh Aman wali chat mein nahi aate.

Yehi concept LangGraph mein Thread ka hai.

In [ ]:
# Why Thread is Needed?

Without Thread

User A
↓

State

↓

User B

↓

Same State ❌

Sab data mix ho jayega.

# With Thread

User A
↓

Thread 1
↓

State A

-------------------

User B
↓

Thread 2
↓

State B

Sabka data separate.

# Relation Between Persistence, Checkpointer and Thread
Persistence
      │
      ▼
Checkpointer
      │
      ▼
Thread
      │
      ▼
State

Persistence → Data ko save rakhna.

Checkpointer → State ko save/load karna.

Thread → Kis user ya kis conversation ka state hai, uski identity.

# A thread in LangGraph is a unique identifier for a workflow or conversation. It keeps the state of each execution separate, allowing multiple users or multiple conversations to run independently without mixing their data.

# InMemorySaver
Is a checkpointer that stores the workflow state in the system's memory (RAM). It is mainly used for development and testing because it is fast, but the saved state is lost when the application stops or restarts.

# Code

In [3]:
!pip install langchain-openai
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 3.4 MB/s eta 0:00:00


In [16]:
import os
from google.colab import userdata

# Load the API key from Colab secrets
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

load_dotenv()

llm = ChatOpenAI()

In [17]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [18]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [19]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [20]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [27]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************Y8YA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

In [23]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1751d4-6956-61b4-8000-571fbf3da5e9'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-07-01T07:20:02.389220+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1751d4-6953-61f2-bfff-de2e6807bcd5'}}, tasks=(PregelTask(id='4c1119c9-ef21-372d-5e2f-d4cd56774393', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error='AuthenticationError("Error code: 401 - {\'error\': {\'message\': \'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************Y8YA. You can find your API key at https://platform.openai.com/account/api-keys.\', \'type\': \'invalid_request_error\', \'code\': \'invalid_api_key\', \'param\': None}, \'status\': 401}")', in

In [24]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1751d4-6956-61b4-8000-571fbf3da5e9'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-07-01T07:20:02.389220+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1751d4-6953-61f2-bfff-de2e6807bcd5'}}, tasks=(PregelTask(id='4c1119c9-ef21-372d-5e2f-d4cd56774393', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error='AuthenticationError("Error code: 401 - {\'error\': {\'message\': \'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************Y8YA. You can find your API key at https://platform.openai.com/account/api-keys.\', \'type\': \'invalid_request_error\', \'code\': \'invalid_api_key\', \'param\': None}, \'status\': 401}")', i

In [28]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************Y8YA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

In [26]:

list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1751d4-6956-61b4-8000-571fbf3da5e9'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-07-01T07:20:02.389220+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1751d4-6953-61f2-bfff-de2e6807bcd5'}}, tasks=(PregelTask(id='4c1119c9-ef21-372d-5e2f-d4cd56774393', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error='AuthenticationError("Error code: 401 - {\'error\': {\'message\': \'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************Y8YA. You can find your API key at https://platform.openai.com/account/api-keys.\', \'type\': \'invalid_request_error\', \'code\': \'invalid_api_key\', \'param\': None}, \'status\': 401}")', i

# Time Travel

In [29]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5"}})

StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f06cc6e-7232-6cb1-8000-f71609e6cec5'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())

In [30]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5"}})

EmptyInputError: Received no input for __start__

In [31]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1751d7-d219-6486-8002-e8392e43664f'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-07-01T07:21:33.904993+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1751d7-d216-6210-8001-89540d82c3f6'}}, tasks=(PregelTask(id='929aa71a-b92a-da71-b2f5-7243c9ce92a8', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error='AuthenticationError("Error code: 401 - {\'error\': {\'message\': \'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************Y8YA. You can find your API key at https://platform.openai.com/account/api-keys.\', \'type\': \'invalid_request_error\', \'code\': \'invalid_api_key\', \'param\': None}, \'status\': 401}")', i

# Updating State

In [32]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1751d8-5aa1-6b8b-8000-a06cfa1fcb88'}}

In [33]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1751d8-5aa1-6b8b-8000-a06cfa1fcb88'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-07-01T07:21:48.221515+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-7232-6cb1-8000-f71609e6cec5'}}, tasks=(PregelTask(id='98413b93-6242-0066-7b2c-5446659d1eaa', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1751d7-d219-6486-8002-e8392e43664f'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-07-01T07:21:33.904993+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': 

In [34]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc72-ca16-6359-8001-7eea05e07dd2"}})

EmptyInputError: Received no input for __start__

In [35]:

list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1751d8-5aa1-6b8b-8000-a06cfa1fcb88'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-07-01T07:21:48.221515+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-7232-6cb1-8000-f71609e6cec5'}}, tasks=(PregelTask(id='98413b93-6242-0066-7b2c-5446659d1eaa', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1751d7-d219-6486-8002-e8392e43664f'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-07-01T07:21:33.904993+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': 

 # Fault Tolerance


In [7]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [8]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [9]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [10]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [13]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
❌ Kernel manually interrupted (crash simulated).


In [36]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)


🔁 Re-running the graph to demonstrate fault tolerance...


AttributeError: 'StateGraph' object has no attribute 'invoke'

In [38]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))

AttributeError: 'StateGraph' object has no attribute 'get_state_history'

# Time Travel

Time Travel ka matlab past mein jaakar workflow ke kisi purane checkpoint se dobara continue ya replay karna.

# Why is Time Travel Use

1. Debugging

Developer dekh sakta hai:

Step 1 ✅

Step 2 ❌ Wrong output

Step 3 Wrong answer

Wapas Step 2 par jaake debug karega.

2. Try Different Paths

Checkpoint
     │
     ├── Path A → Answer A
     │
     └── Path B → Answer B

Ek hi checkpoint se multiple experiments kar sakte ho.

3. Human-in-the-Loop

Suppose AI ne ye approve kiya:

Discount = 5%

Manager bolta hai:

"Nahi, 20% discount do."

Time travel:

Go back
↓

Update state

↓

Continue

Workflow ko dobara start nahi karna padta.

# Fault Tolerance

Agar workflow ke beech mein koi error, crash ya failure aa jaye, to system usko handle karke bina sara kaam dobara kiye continue kar sake.

Simple words mein:

"System fail hone ke baad bhi recover karke kaam continue karna."

# Real-Life Example 🚗

Socho tum car se Delhi se Jaipur ja rahe ho.

Raste mein tyre puncture ho gaya.

Without Fault Tolerance

Delhi
  │
  ▼
💥 Tyre puncture

Go back to Delhi ❌

Start journey again

# Why is Fault Tolerance Important?

Imagine:

LLM API is temporarily unavailable.

Database connection breaks.

Server crashes.

Network disconnects.

* Without fault tolerance:

Everything starts from the beginning.

* With fault tolerance:

Recover from the failure and continue from the last checkpoint.

In [ ]:
# Relation with Persistence
Persistence
      │
      ▼
Checkpointer
      │
      ▼
Saved State
      │
      ▼
Fault Tolerance

Persistence stores the workflow state.

Checkpointer saves and restores that state.

Fault tolerance uses the saved state to recover after failures.

# Fault tolerance is the ability of a LangGraph workflow to recover from failures such as crashes, network issues, or node errors by using persisted checkpoints, allowing execution to continue from the last successful step instead of restarting the entire workflow.